#### Part A - Handling Missing Values

In [115]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import SimpleImputer,KNNImputer, MissingIndicator, IterativeImputer

In [116]:
# Load dataset

data = pd.read_csv('patient_health_records.csv')
data.head()

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,P0001,69.0,Male,South,29.140042,119.967503,175.880159,87.704018,1
1,P0002,NaN,Male,West,32.240988,139.080885,233.505779,112.649952,0
2,P0003,89.0,Female,East,27.150680,102.695640,218.580412,102.578273,0
3,P0004,78.0,Male,North,23.074774,121.704367,244.911816,113.778262,0
4,P0005,38.0,Male,West,25.785560,114.053463,205.870489,115.380798,0


In [117]:
# INFO

print(f'Missing values percentage: \n{
data.isnull().sum() / len(data) *100}')

Missing values percentage: 
patient_id         0.0
age               10.0
gender            10.0
region            10.0
bmi                9.8
blood_pressure     0.0
cholesterol        9.6
glucose            9.8
disease_risk       0.0
dtype: float64


In [118]:
print(f'Describe: \n{data.describe()}')

Describe: 
              age         bmi  blood_pressure  cholesterol     glucose  \
count  900.000000  902.000000     1000.000000   904.000000  902.000000   
mean    52.708889   26.839617      122.977126   191.914894  108.413610   
std     20.954363    5.783429       21.986566    45.789680   45.244413   
min     18.000000   10.211299       69.907497    80.776090   53.023842   
25%     34.750000   23.799395      110.407875   166.674909   91.102469   
50%     52.000000   26.741862      120.713856   189.485116  100.740439   
75%     71.000000   29.480797      131.710686   211.606497  111.265799   
max     89.000000   64.421470      239.844896   497.142770  393.786076   

       disease_risk  
count   1000.000000  
mean       0.422000  
std        0.494126  
min        0.000000  
25%        0.000000  
50%        0.000000  
75%        1.000000  
max        1.000000  


In [119]:
print(f'Info: \n{data.info()}')

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   patient_id      1000 non-null   str    
 1   age             900 non-null    float64
 2   gender          900 non-null    str    
 3   region          900 non-null    str    
 4   bmi             902 non-null    float64
 5   blood_pressure  1000 non-null   float64
 6   cholesterol     904 non-null    float64
 7   glucose         902 non-null    float64
 8   disease_risk    1000 non-null   int64  
dtypes: float64(5), int64(1), str(3)
memory usage: 70.4 KB
Info: 
None


In [120]:
# SimpleImputer

data_simp = data.copy()

mean_imp = SimpleImputer(strategy='mean')
median_imp = SimpleImputer(strategy='median')
freq_imp = SimpleImputer(strategy='most_frequent')

data_simp['bmi'] = mean_imp.fit_transform(data_simp[['bmi']])

data_simp[['age','cholesterol','glucose']] = median_imp.fit_transform(data_simp[['age','cholesterol','glucose']])

data_simp[['gender','region']] = freq_imp.fit_transform(data_simp[['gender','region']])

In [121]:
# Missing Indicator + Random Sample Imputation

data_rand = data.copy()

def random_sample_imp(df ,col, random_state=9009):
    np.random.seed(random_state)

    null_count = df[col].isnull().sum()

    if null_count > 0:
        random_value = df[col].dropna().sample(
            n=null_count,
            replace=True,
            random_state=random_state
        ).values

        df_ = df.copy()
        df_.loc[df_[col].isnull(), col] = random_value

        return df_

    return df


In [122]:
indicator = MissingIndicator(features='missing-only')
indicator_array = indicator.fit_transform(data)

indicator_col = [data.columns[col] + '_Missing' for col in indicator.features_]
indicator_df = pd.DataFrame(indicator_array.astype(int), columns=indicator_col)
indicator_df

,age_Missing,gender_Missing,region_Missing,bmi_Missing,cholesterol_Missing,glucose_Missing
0,0,0,0,0,0,0
1,1,0,0,0,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0
...,...,...,...,...,...,...
995,0,0,0,0,0,0
996,1,0,0,0,0,0
997,0,0,0,0,0,0
998,0,0,0,1,0,1


In [123]:
imputed_df = data.copy()

for col in data.columns:
    imputed_df = random_sample_imp(imputed_df,col)

imputed_df

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,P0001,69.0,Male,South,29.140042,119.967503,175.880159,87.704018,1
1,P0002,74.0,Male,West,32.240988,139.080885,233.505779,112.649952,0
2,P0003,89.0,Female,East,27.150680,102.695640,218.580412,102.578273,0
3,P0004,78.0,Male,North,23.074774,121.704367,244.911816,113.778262,0
4,P0005,38.0,Male,West,25.785560,114.053463,205.870489,115.380798,0
...,...,...,...,...,...,...,...,...,...
995,P0996,27.0,Female,North,30.797070,112.479519,194.004461,132.418363,0
996,P0997,87.0,Female,South,24.818420,105.567162,179.760156,90.737693,0
997,P0998,72.0,Female,East,31.950468,127.536709,189.653461,98.883793,0
998,P0999,49.0,Male,North,21.101805,117.615964,169.801326,111.850220,0


In [124]:
data_rand = pd.concat([imputed_df, indicator_df], axis=1)
data_rand

,patient_id,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk,age_Missing,gender_Missing,region_Missing,bmi_Missing,cholesterol_Missing,glucose_Missing
0,P0001,69.0,Male,South,29.140042,119.967503,175.880159,87.704018,1,0,0,0,0,0,0
1,P0002,74.0,Male,West,32.240988,139.080885,233.505779,112.649952,0,1,0,0,0,0,0
2,P0003,89.0,Female,East,27.150680,102.695640,218.580412,102.578273,0,0,0,0,0,0,0
3,P0004,78.0,Male,North,23.074774,121.704367,244.911816,113.778262,0,0,0,0,0,0,0
4,P0005,38.0,Male,West,25.785560,114.053463,205.870489,115.380798,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,P0996,27.0,Female,North,30.797070,112.479519,194.004461,132.418363,0,0,0,0,0,0,0
996,P0997,87.0,Female,South,24.818420,105.567162,179.760156,90.737693,0,1,0,0,0,0,0
997,P0998,72.0,Female,East,31.950468,127.536709,189.653461,98.883793,0,0,0,0,0,0,0
998,P0999,49.0,Male,North,21.101805,117.615964,169.801326,111.850220,0,0,0,0,1,0,1


In [125]:
# KNNImputer

data_knn = data.copy()
data_knn_id = data_knn['patient_id'].copy()
data_knn.drop(['patient_id'], axis=1, inplace=True)

In [126]:
data_knn['gender']= data_knn['gender'].map({'Male':0, 'Female':1})
data_knn['region'] = data_knn['region'].map({'East':0, 'West':1, 'North':2, 'South':3})

In [127]:
knn = KNNImputer(n_neighbors=5, weights='distance')

data_knn = pd.DataFrame(knn.fit_transform(data_knn),columns=data_knn.columns)
data_knn

,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk
0,69.000000,0.0,3.0,29.140042,119.967503,175.880159,87.704018,1.0
1,57.410974,0.0,1.0,32.240988,139.080885,233.505779,112.649952,0.0
2,89.000000,1.0,0.0,27.150680,102.695640,218.580412,102.578273,0.0
3,78.000000,0.0,2.0,23.074774,121.704367,244.911816,113.778262,0.0
4,38.000000,0.0,1.0,25.785560,114.053463,205.870489,115.380798,0.0
...,...,...,...,...,...,...,...,...
995,27.000000,1.0,2.0,30.797070,112.479519,194.004461,132.418363,0.0
996,68.345275,1.0,3.0,24.818420,105.567162,179.760156,90.737693,0.0
997,72.000000,1.0,0.0,31.950468,127.536709,189.653461,98.883793,0.0
998,49.000000,0.0,2.0,24.586742,117.615964,169.801326,100.488883,0.0


In [128]:
data_knn['gender'] = data_knn['gender'].round().astype(int)
data_knn['region'] = data_knn['region'].round().astype(int)

data_knn['gender']= data_knn['gender'].map({0:'Male', 1:'Female'})
data_knn['region'] = data_knn['region'].map({0:'East', 1:'West', 2:'North', 3:'South'})

In [129]:
data_knn = pd.concat([data_knn, data_knn_id], axis=1)
data_knn

,age,gender,region,bmi,blood_pressure,cholesterol,glucose,disease_risk,patient_id
0,69.000000,Male,South,29.140042,119.967503,175.880159,87.704018,1.0,P0001
1,57.410974,Male,West,32.240988,139.080885,233.505779,112.649952,0.0,P0002
2,89.000000,Female,East,27.150680,102.695640,218.580412,102.578273,0.0,P0003
3,78.000000,Male,North,23.074774,121.704367,244.911816,113.778262,0.0,P0004
4,38.000000,Male,West,25.785560,114.053463,205.870489,115.380798,0.0,P0005
...,...,...,...,...,...,...,...,...,...
995,27.000000,Female,North,30.797070,112.479519,194.004461,132.418363,0.0,P0996
996,68.345275,Female,South,24.818420,105.567162,179.760156,90.737693,0.0,P0997
997,72.000000,Female,East,31.950468,127.536709,189.653461,98.883793,0.0,P0998
998,49.000000,Male,North,24.586742,117.615964,169.801326,100.488883,0.0,P0999
